In [ ]:
!wget -O banglaclip_model_epoch_10.pth https://huggingface.co/Mansuba/BanglaCLIP13/resolve/main/banglaclip_model_epoch_10.pth


--2025-01-21 11:54:08--  https://huggingface.co/Mansuba/BanglaCLIP13/resolve/main/banglaclip_model_epoch_10.pth
Resolving huggingface.co (huggingface.co)... 18.164.174.17, 18.164.174.118, 18.164.174.23, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.17|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cdn-lfs-us-1.hf.co/repos/2a/ad/2aadee49f0f0c86b7b267a72f0937b06e9c98f694b60f9d09a1698caad62340d/f25c01d0773579e603903fefc52f721337cf92cbcbfb4ab6e48d2c858c8cbc3f?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27banglaclip_model_epoch_10.pth%3B+filename%3D%22banglaclip_model_epoch_10.pth%22%3B&Expires=1737464048&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTczNzQ2NDA0OH19LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy11cy0xLmhmLmNvL3JlcG9zLzJhL2FkLzJhYWRlZTQ5ZjBmMGM4NmI3YjI2N2E3MmYwOTM3YjA2ZTljOThmNjk0YjYwZjlkMDlhMTY5OGNhYWQ2MjM0MGQvZjI1YzAxZDA3NzM1NzllNjAzOTAzZmVmYzUyZjcyMTMzN2NmOTJjYmNiZmI0YW

In [ ]:
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.4/321.4 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 117.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 5.5 MB/s eta 0:00:00
  Attempting uninstall: markupsafe
    Found existing installation: MarkupSafe 3.0.2
    Uninstalling MarkupSafe-3.0.2:
      Successfully uninstalled MarkupSafe-3.0.2


In [ ]:
import torch
from transformers import CLIPModel, CLIPProcessor, AutoTokenizer, MarianMTModel, MarianTokenizer
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
import numpy as np
from typing import List, Tuple, Optional
import gradio as gr

class EnhancedBanglaSDGenerator:
    def __init__(self, banglaclip_weights_path: str, device: Optional[torch.device] = None):
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Initialize models and processors
        self.clip_model_name = "openai/clip-vit-base-patch32"
        self.bangla_text_model = "csebuetnlp/banglabert"

        # Initialize translation models
        self.bn2en_model_name = "Helsinki-NLP/opus-mt-bn-en"
        self.translator = MarianMTModel.from_pretrained(self.bn2en_model_name).to(self.device)
        self.trans_tokenizer = MarianTokenizer.from_pretrained(self.bn2en_model_name)

        # Load BanglaCLIP with improved initialization
        self.banglaclip_model = self._load_banglaclip_model(banglaclip_weights_path)
        self.processor = CLIPProcessor.from_pretrained(self.clip_model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(self.bangla_text_model)

        # Enhanced Stable Diffusion initialization
        self.pipe = StableDiffusionPipeline.from_pretrained(
            "runwayml/stable-diffusion-v1-5",
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            safety_checker=None
        )
        self.pipe.scheduler = DPMSolverMultistepScheduler.from_config(
            self.pipe.scheduler.config,
            use_karras_sigmas=True,
            algorithm_type="dpmsolver++"
        )
        self.pipe = self.pipe.to(self.device)

        # Move models to device
        self.banglaclip_model = self.banglaclip_model.to(self.device)

        # Initialize location and scene contexts
        self.location_contexts = {
            'কক্সবাজার': 'Cox\'s Bazar beach, longest natural sea beach in the world, sandy beach',
            'সেন্টমার্টিন': 'Saint Martin\'s Island, coral island, tropical paradise',
            'সুন্দরবন': 'Sundarbans mangrove forest, Bengal tigers, riverine forest',
            'কুয়াকাটা': 'Kuakata beach, sandy beach, Bay of Bengal',
            'সাজেক': 'Sajek Valley, hill tract, misty mountains',
            'বান্দরবান': 'Bandarban hills, tribal area, mountain ranges',
            'রাঙ্গামাটি': 'Rangamati lake, hill district, water body',
            'শ্রীমঙ্গল': 'Sreemangal tea gardens, tea estates, rolling hills'
        }

        self.scene_contexts = {
            'সৈকত': 'beach, seaside, waves, sandy shore, ocean view',
            'সমুদ্র': 'ocean, sea waves, deep blue water, horizon',
            'পাহাড়': 'mountains, hills, valleys, scenic landscape',
            'জলপ্রপাত': 'waterfall, cascading water, natural formation',
            'নদী': 'river, flowing water, riverbank, water body',
            'বন': 'forest, woods, trees, natural habitat',
            'গ্রাম': 'village, rural landscape, countryside',
            'শহর': 'city, urban landscape, buildings',
            'মন্দির': 'temple, religious architecture, sacred place',
            'মসজিদ': 'mosque, Islamic architecture, religious place'
        }

    def _load_banglaclip_model(self, weights_path: str) -> CLIPModel:
        try:
            clip_model = CLIPModel.from_pretrained(self.clip_model_name)
            state_dict = torch.load(weights_path, map_location=self.device)

            cleaned_state_dict = {}
            for k, v in state_dict.items():
                k = k.replace('module.', '')
                k = k.replace('clip.', '')
                if k.startswith('text_model.') or k.startswith('vision_model.'):
                    cleaned_state_dict[k] = v

            clip_model.load_state_dict(cleaned_state_dict, strict=False)
            return clip_model
        except Exception as e:
            raise RuntimeError(f"Failed to load BanglaCLIP model: {str(e)}")

    def _translate_text(self, bangla_text: str) -> str:
        try:
            # Tokenize the Bangla text
            inputs = self.trans_tokenizer(bangla_text, return_tensors="pt", padding=True).to(self.device)

            # Generate translation
            with torch.no_grad():
                translated = self.translator.generate(**inputs)

            # Decode the translation
            translated_text = self.trans_tokenizer.decode(translated[0], skip_special_tokens=True)
            return translated_text
        except Exception as e:
            print(f"Translation error: {str(e)}")
            return bangla_text

    def _get_text_embedding(self, bangla_text: str) -> torch.Tensor:
        inputs = self.tokenizer(
            bangla_text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=77,
            return_attention_mask=True
        ).to(self.device)

        inputs.pop("token_type_ids", None)

        with torch.no_grad():
            text_features = self.banglaclip_model.get_text_features(**inputs)
            text_features = text_features / text_features.norm(dim=-1, keepdim=True)

        return text_features

    def _enhance_prompt(self, bangla_text: str) -> str:
        # Get BanglaCLIP embedding
        text_features = self._get_text_embedding(bangla_text)

        # Get translation
        translated_text = self._translate_text(bangla_text)

        # Initialize context parts
        context_parts = []

        # Check for specific locations and scenes from BanglaCLIP context
        for location, context in self.location_contexts.items():
            if location in bangla_text:
                context_parts.append(context)

        for scene, context in self.scene_contexts.items():
            if scene in bangla_text:
                context_parts.append(context)

        # Add time context based on translated text
        time_keywords = {
            "morning": "sunrise, dawn, golden hour",
            "evening": "sunset, dusk, golden hour",
            "night": "moonlight, stars, evening sky",
            "afternoon": "daylight, bright sun",
            "dawn": "sunrise, early morning light",
            "dusk": "sunset, twilight"
        }

        for time_key, time_desc in time_keywords.items():
            if time_key.lower() in translated_text.lower():
                context_parts.append(time_desc)

        # Photography style enhancements
        photo_style = [
            "professional photography",
            "high resolution",
            "4k",
            "detailed",
            "realistic",
            "beautiful composition",
            "sharp focus",
            "HDR",
            "dramatic lighting",
            "cinematic view"
        ]

        # Combine BanglaCLIP and translation results
        enhanced_parts = [translated_text] + context_parts + photo_style

        # Remove duplicates while maintaining order
        seen = set()
        enhanced_parts = [x for x in enhanced_parts if not (x in seen or seen.add(x))]

        enhanced_prompt = ", ".join(enhanced_parts)
        return enhanced_prompt

    def generate_image(
        self,
        bangla_text: str,
        num_images: int = 1,
        num_inference_steps: int = 50,
        guidance_scale: float = 7.5,
        seed: Optional[int] = None
    ) -> Tuple[List[any], str]:
        try:
            if seed is not None:
                torch.manual_seed(seed)

            enhanced_prompt = self._enhance_prompt(bangla_text)

            negative_prompt = (
                "blurry, low quality, pixelated, cartoon, anime, illustration, "
                "painting, drawing, artificial, fake, oversaturated, undersaturated, "
                "distorted, deformed, bad anatomy, bad proportions, watermark, "
                "signature, text, bad composition, extra limbs, duplicate, multiple"
            )

            with torch.autocast(self.device.type):
                result = self.pipe(
                    prompt=enhanced_prompt,
                    negative_prompt=negative_prompt,
                    num_images_per_prompt=num_images,
                    num_inference_steps=num_inference_steps,
                    guidance_scale=guidance_scale
                )

            return result.images, enhanced_prompt

        except Exception as e:
            print(f"Error during image generation: {str(e)}")
            return None, None

def generate_images(text, num_images, steps, guidance_scale, seed):
    if not text.strip():
        return None, "দয়া করে কিছু টেক্সট লিখুন"

    images, prompt = generator.generate_image(
        bangla_text=text,
        num_images=int(num_images),
        num_inference_steps=int(steps),
        guidance_scale=float(guidance_scale),
        seed=int(seed) if seed else None
    )

    if images:
        return images, prompt
    return None, "ছবি তৈরি ব্যর্থ হয়েছে"

# Initialize the generator
generator = EnhancedBanglaSDGenerator(
    banglaclip_weights_path="banglaclip_model_epoch_10.pth"
)

# Create Gradio interface
demo = gr.Interface(
    fn=generate_images,
    inputs=[
        gr.Textbox(
            label="বাংলা টেক্সট লিখুন",
            placeholder="যেকোনো বাংলা টেক্সট লিখুন...",
            lines=3
        ),
        gr.Slider(
            minimum=1,
            maximum=4,
            step=1,
            value=1,
            label="ছবির সংখ্যা"
        ),
        gr.Slider(
            minimum=20,
            maximum=100,
            step=1,
            value=50,
            label="স্টেপস"
        ),
        gr.Slider(
            minimum=1.0,
            maximum=20.0,
            step=0.5,
            value=7.5,
            label="গাইডেন্স স্কেল"
        ),
        gr.Number(
            label="সীড (ঐচ্ছিক)",
            precision=0
        )
    ],
    outputs=[
        gr.Gallery(label="তৈরি করা ছবি"),
        gr.Textbox(label="ব্যবহৃত প্রম্পট")
    ],
    title="বাংলা টেক্সট থেকে ছবি তৈরি",
    description="যেকোনো বাংলা টেক্সট দিয়ে উচ্চমানের ছবি তৈরি করুন"
)

if __name__ == "__main__":
    demo.launch(share=True)

config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/309M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/2.25M [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
<ipython-input-7-9b883a0000fe>:70: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full contr

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://df86232f52ce1b99fb.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


#1st test

In [ ]:
import torch
from transformers import CLIPModel, CLIPProcessor, AutoTokenizer
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
import numpy as np
from typing import List, Tuple, Optional
import gradio as gr

class EnhancedBanglaSDGenerator:
    def __init__(self, banglaclip_weights_path: str, device: Optional[torch.device] = None):
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Initialize models and processors
        self.clip_model_name = "openai/clip-vit-base-patch32"
        self.bangla_text_model = "csebuetnlp/banglabert"

        # Load BanglaCLIP with improved initialization
        self.banglaclip_model = self._load_banglaclip_model(banglaclip_weights_path)
        self.processor = CLIPProcessor.from_pretrained(self.clip_model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(self.bangla_text_model)

        # Enhanced Stable Diffusion initialization
        self.pipe = StableDiffusionPipeline.from_pretrained(
            "runwayml/stable-diffusion-v1-5",
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            safety_checker=None
        )
        self.pipe.scheduler = DPMSolverMultistepScheduler.from_config(
            self.pipe.scheduler.config,
            use_karras_sigmas=True,
            algorithm_type="dpmsolver++"
        )
        self.pipe = self.pipe.to(self.device)

        # Move models to device
        self.banglaclip_model = self.banglaclip_model.to(self.device)

        # Initialize translation mapping
        self.translation_map = {
            'প্রাকৃতিক': 'natural',
            'সুন্দর': 'beautiful',
            'দৃশ্য': 'scene',
            'পাহাড়': 'mountain',
            'সূর্য': 'sun',
            'আকাশ': 'sky',
            'মেঘ': 'cloud',
            'নদী': 'river',
            'সমুদ্র': 'ocean',
            'গাছ': 'tree',
            'ফুল': 'flower'
        }

    def _load_banglaclip_model(self, weights_path: str) -> CLIPModel:
        try:
            clip_model = CLIPModel.from_pretrained(self.clip_model_name)
            state_dict = torch.load(weights_path, map_location=self.device)

            cleaned_state_dict = {}
            for k, v in state_dict.items():
                k = k.replace('module.', '')
                k = k.replace('clip.', '')
                if k.startswith('text_model.') or k.startswith('vision_model.'):
                    cleaned_state_dict[k] = v

            clip_model.load_state_dict(cleaned_state_dict, strict=False)
            return clip_model
        except Exception as e:
            raise RuntimeError(f"Failed to load BanglaCLIP model: {str(e)}")

    def _get_text_embedding(self, bangla_text: str) -> torch.Tensor:
        inputs = self.tokenizer(
            bangla_text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=77,
            return_attention_mask=True
        ).to(self.device)

        inputs.pop("token_type_ids", None)

        with torch.no_grad():
            text_features = self.banglaclip_model.get_text_features(**inputs)
            text_features = text_features / text_features.norm(dim=-1, keepdim=True)

        return text_features

    def _enhance_prompt_with_clip(self, text_features: torch.Tensor, original_text: str) -> str:
        words = original_text.split()
        translated_parts = []
        for word in words:
            if word in self.translation_map:
                translated_parts.append(self.translation_map[word])
            else:
                translated_parts.append(word)

        base_translation = " ".join(translated_parts)
        style_keywords = ["high quality", "detailed", "professional photography", "4k", "sharp focus"]

        if any(word in original_text for word in ['প্রাকৃতিক', 'দৃশ্য']):
            style_keywords.extend([
                "landscape photography",
                "dramatic lighting",
                "golden hour",
                "cinematic",
                "high resolution"
            ])

        if any(word in original_text for word in ['সূর্য', 'আকাশ']):
            style_keywords.extend([
                "atmospheric",
                "beautiful sky",
                "natural lighting",
                "HDR",
                "vivid colors"
            ])

        enhanced_prompt = f"{base_translation}, {', '.join(style_keywords)}"
        return enhanced_prompt

    def generate_image(
        self,
        bangla_text: str,
        num_images: int = 1,
        num_inference_steps: int = 50,
        guidance_scale: float = 8.5,
        seed: Optional[int] = None
    ) -> Tuple[List[any], str]:
        try:
            if seed is not None:
                torch.manual_seed(seed)

            text_features = self._get_text_embedding(bangla_text)
            enhanced_prompt = self._enhance_prompt_with_clip(text_features, bangla_text)

            negative_prompt = (
                "blurry, low quality, pixelated, cartoon, fuzzy, noisy, "
                "oversaturated, undersaturated, distorted, deformed, "
                "bad anatomy, watermark, signature, text"
            )

            with torch.autocast(self.device.type):
                result = self.pipe(
                    prompt=enhanced_prompt,
                    negative_prompt=negative_prompt,
                    num_images_per_prompt=num_images,
                    num_inference_steps=num_inference_steps,
                    guidance_scale=guidance_scale
                )

            return result.images, enhanced_prompt

        except Exception as e:
            print(f"Error during image generation: {str(e)}")
            return None, None

def generate_images(text, num_images, steps, guidance_scale, seed):
    if not text.strip():  # Check if text is empty or only whitespace
        return None, "Please enter some text"

    images, prompt = generator.generate_image(
        bangla_text=text,
        num_images=int(num_images),
        num_inference_steps=int(steps),
        guidance_scale=float(guidance_scale),
        seed=int(seed) if seed else None
    )

    if images:
        return images, prompt
    return None, "Generation failed"

# Initialize the generator
generator = EnhancedBanglaSDGenerator(
    banglaclip_weights_path="banglaclip_model_epoch_10.pth"
)

# Create Gradio interface
demo = gr.Interface(
    fn=generate_images,
    inputs=[
        gr.Textbox(
            label="Enter Bangla Text",
            placeholder="Write your text here...",
            lines=3
        ),
        gr.Slider(
            minimum=1,
            maximum=4,
            step=1,
            value=1,
            label="Number of Images"
        ),
        gr.Slider(
            minimum=20,
            maximum=100,
            step=1,
            value=50,
            label="Steps"
        ),
        gr.Slider(
            minimum=1.0,
            maximum=20.0,
            step=0.5,
            value=8.5,
            label="Guidance Scale"
        ),
        gr.Number(
            label="Seed (optional)",
            precision=0
        )
    ],
    outputs=[
        gr.Gallery(label="Generated Images"),
        gr.Textbox(label="Generated Prompt")
    ],
    title="Bangla Text to Image Generator",
    description="Enter any Bangla text to generate images"
)

if __name__ == "__main__":
    # Launch with sharing enabled for testing
    demo.launch(share=True)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

<ipython-input-4-d38dec77615a>:55: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(weights_path, map_location=self.device)


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

...

unet/config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://892b4d561095e2321c.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
